# Helmet & Safety Vest Detection: training and evaluation workflow

This notebook reproduces the whole workflow in **11 numbered sections** (data, training, evaluation, the Safe / Not-safe logic).
Run them **top to bottom** on a Colab **T4 GPU**. Every section says which file of the GitHub repo it relates to, and **every code cell is
explained with comments** (`#` lines). Read the comments first, then run the cell, then look at the output.

Repo: https://github.com/suhaniranka006/helmet-vest-detection

**Setup before you start:** Runtime > Change runtime type > **T4 GPU**. In the left sidebar (key icon, "Secrets") add `ROBOFLOW_API_KEY` and switch on *Notebook access*.

| Section | What it does | Related repo file |
|---|---|---|
| 1 | GPU check and installs | `requirements.txt` |
| 2 | Download the dataset | (Colab only) |
| 3 | Inspect the data | (Colab only) |
| 4 | Train YOLOv8n | (Colab only). Result = `models/best_n.pt` |
| 5 | Look at the training curves | (Colab only) |
| 6 | Evaluate on the TEST set + confusion matrix | numbers are in `README.md` |
| 7 | (Optional) compare with YOLOv8s | numbers are in `README.md` |
| 8 | Save the best weights | `models/best_n.pt` |
| 9 | Predict on one image and read the output | what `app.py` does inside |
| 10 | The full pipeline with the repo's own logic | `verdict.py`, `app.py` |
| 11 | Ground-truth evaluation | `eval/run_eval.py`, `eval/metrics.py` |

**How to read the comments:** lines starting with `# WHY:` give the reason, `# NOTE:` give a trap or detail,
`# TRY:` is a small experiment.

## 1. Setup: GPU and libraries
*Why:* training needs a GPU. `ultralytics` contains YOLOv8; `roboflow` downloads datasets.

In [ ]:
# WHY: show which GPU Colab gave us. You should see "Tesla T4".
# NOTE: if you see "nvidia-smi: command not found", you are on a CPU runtime and training will take
#       HOURS instead of ~13 minutes. Fix: Runtime > Change runtime type > T4 GPU > Save, then run again.
# (a line starting with ! runs a Linux shell command, not Python)
!nvidia-smi

# WHY: install the two libraries we need.
#  - ultralytics = the library that contains YOLOv8 (training and prediction)
#  - roboflow    = lets us download the dataset by name
# -q means "quiet" (less text printed)
# NOTE: ultralytics==8.4.215 is the exact version my results were made with ("pinning" = reproducible).
!pip -q install ultralytics==8.4.215 roboflow

## 2. Download the dataset (Colab only)
*Why:* the data lives on Roboflow. The API key comes from Colab **Secrets**, never from the code.
Replace the workspace, project and version number with the values from your own Roboflow project URL
(`app.roboflow.com/WORKSPACE/PROJECT/VERSION`).

**Note:** the workspace and project names below are my own fork. The original public dataset is "vest" by OAKDev (CC BY 4.0): https://universe.roboflow.com/oakdev/vest-pbrbu . Fork it into your own Roboflow account and use your own names, version and API key.

In [ ]:
# userdata gives access to the Colab Secrets (the key icon in the left sidebar)
from google.colab import userdata
# Roboflow is the class that connects to the Roboflow website
from roboflow import Roboflow

# WHY: read the key from Secrets so it is never written inside the code.
# NOTE: code goes to GitHub; a key inside code = leaked password. Never print this variable.
api_key = userdata.get("ROBOFLOW_API_KEY")

# log in to Roboflow with that key
rf = Roboflow(api_key=api_key)

# pick MY copy of the dataset: workspace name -> project name
# (you can read both names in the project URL on app.roboflow.com)
project = rf.workspace("suhanis-workspace-0lb1u").project("vest-pbrbu-thbgy")

# download version 1 in "yolov8" format (images + one .txt label file per image + data.yaml)
dataset = project.version(1).download("yolov8")

# the folder where the data was saved, e.g. /content/vest-1. We reuse this in later cells.
print(dataset.location)

## 3. Inspect the data (Colab only)
*Why:* wrong paths or class order are the most common silent bugs. Always look before you train.
Expected here: train 959, valid 268, test 132 images, and 4 classes.

In [ ]:
import os, glob      # os = work with folders and files, glob = find files by pattern

# base = the dataset folder from section 2
base = dataset.location

# count the images in each split (the three folders train / valid / test)
for split in ["train", "valid", "test"]:
    # os.listdir lists the names inside a folder; len() counts them
    n = len(os.listdir(f"{base}/{split}/images"))
    print(split, "->", n, "images")

# data.yaml tells YOLO where the images are and what the class numbers mean.
# NOTE: the ORDER of names matters: class 0 = first name, class 1 = second, and so on.
print(open(f"{base}/data.yaml").read())

# take the first label file in the train folder to see what a label looks like
label_file = glob.glob(f"{base}/train/labels/*.txt")[0]
print(label_file)

# each line = one box:  class_number  x_center  y_center  width  height
# all four numbers are between 0 and 1, RELATIVE to the image size (so they survive resizing)
print(open(label_file).read())

## 4. Train YOLOv8n (Colab only)
*Why:* fine-tuning a COCO-pretrained model on our images is faster and needs less data than training from scratch.
`epochs=50` at most, `patience=10` = stop if the validation score does not improve for 10 epochs.
Takes about 13 minutes on a T4. Output goes to `/content/runs/helmet_vest_v1`.

In [ ]:
# YOLO is the main class of the ultralytics library
from ultralytics import YOLO

# WHY: start from the nano model that is ALREADY trained on COCO (over a hundred thousand images).
# That is transfer learning / fine-tuning. The file is downloaded automatically.
# "n" = nano = the smallest and fastest version.
model = YOLO("yolov8n.pt")

# model.train() is the whole training loop: load a batch, augment, predict,
# compute loss, update weights, validate after each epoch, save weights, early-stop.
results = model.train(
    data=f"{base}/data.yaml",   # where the data is and what the classes are
    epochs=50,                  # at most 50 passes over the training data (1 pass = 1 epoch)
    imgsz=640,                  # every image is resized to 640 x 640
    batch=16,                   # 16 images are processed before each weight update
    patience=10,                # early stopping: stop if validation does not improve for 10 epochs
    project="/content/runs",    # parent output folder
    name="helmet_vest_v1",      # output goes to /content/runs/helmet_vest_v1
    exist_ok=True,              # NOTE: without this, a 2nd run makes helmet_vest_v1-2, -3 ... and the
                                # paths in the next cells (weights, results.png) would point to the OLD run
)
# While it runs, watch the table: box_loss / cls_loss / dfl_loss should FALL, mAP50 should RISE.

## 5. Look at the training curves
*Why:* this is how you *see* overfitting and early stopping. Losses should fall; validation mAP should rise and then flatten.

In [ ]:
# Image + display show a picture inside the notebook
from IPython.display import Image, display

# results.png = the curves of every loss and metric, epoch by epoch (YOLO saves it for us).
# WHY: if training loss keeps falling but validation mAP stops rising, that is overfitting.
display(Image("/content/runs/helmet_vest_v1/results.png", width=900))

## 6. Evaluate on the TEST set and look at the confusion matrix
*Why:* the test set was never used for any decision, so this is the honest score. Use `best.pt` (the best epoch), not `last.pt`.
Reported number in the README: mAP50 about 0.72 on the 132 test images.

In [ ]:
# WHY best.pt: it is the epoch with the best VALIDATION score (epoch 39 in my run), not the last epoch.
best = YOLO("/content/runs/helmet_vest_v1/weights/best.pt")

# split="test" = score on the 132 images the model never used for any decision.
# It prints a table: Precision, Recall, mAP50, mAP50-95 for all classes and for each class.
# NOTE: always quote the TEST numbers. The validation score is slightly optimistic
#       because we picked the best epoch using it.
metrics = best.val(data=f"{base}/data.yaml", split="test",
                   project="/content/runs", name="test_eval", exist_ok=True)

In [ ]:
# The confusion matrix: rows = what the model predicted, columns = the true class.
# The diagonal = correct. "background" = nothing was there (misses and false alarms).
# NOTE: this uses a fixed confidence threshold, so its numbers differ a little from the table.
display(Image("/content/runs/test_eval/confusion_matrix_normalized.png", width=700))

## 7. (Optional) Compare with a bigger model, YOLOv8s
*Why:* same settings, only the model changes. If a bigger model does not help, the limit is the data, not the model size.
This takes about 15 more minutes. Skip it if you are short on time.

In [ ]:
# WHY: a FAIR comparison = same data, same epochs, same patience, same imgsz. Only the model size changes.
# "s" = small: about 11 million parameters (the nano model has about 3 million).
model_s = YOLO("yolov8s.pt")
model_s.train(data=f"{base}/data.yaml", epochs=50, imgsz=640, batch=16, patience=10,
              project="/content/runs", name="helmet_vest_s", exist_ok=True)

# evaluate it on the SAME test set as the nano model. Never compare one model's validation
# score with the other model's test score.
best_s = YOLO("/content/runs/helmet_vest_s/weights/best.pt")
best_s.val(data=f"{base}/data.yaml", split="test", project="/content/runs", name="test_eval_s", exist_ok=True)
# My result: mAP50 0.725 (v8s) vs 0.718 (v8n) = no real gain, so data is the bottleneck.

## 8. Save the best weights
*Why:* Colab erases everything when the session ends. `best.pt` is the file that ended up as `models/best_n.pt` in the repo.

In [ ]:
import shutil                          # shutil.copy copies a file
from google.colab import files          # files.download sends a file to your laptop

# copy it to a clear name FIRST. NOTE: I once downloaded two models both named best.pt and one
# overwrote the other. Name files by what they are: best_n.pt (nano), best_s.pt (small).
shutil.copy("/content/runs/helmet_vest_v1/weights/best.pt", "/content/best_n.pt")
files.download("/content/best_n.pt")

## 9. Predict on ONE image and read the output
*Why:* this is the object that `app.py` works with. The cell downloads one sample photo for you; you can switch to your own photo (see the comments).
`xyxy` = pixel corners (x1, y1, x2, y2), `cls` = class number, `conf` = confidence, `names` = number to name.

In [ ]:
# WHY: the model needs a real image FILE. If the path does not exist you get
#      "FileNotFoundError: ... does not exist" (the file is simply not there).
# Option A (default): download one sample worker photo from my public GitHub repo. No upload needed.
import urllib.request
IMAGE = "/content/my_image.jpg"
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/suhaniranka006/helmet-vest-detection/main/samples/18111488.jpg",
    IMAGE)
# Option B: use your own photo. Click the folder icon in the left sidebar, upload it, then set
#           IMAGE = "/content/<your file name>"  (delete the download lines above).

# model.predict(...) runs the detector on the image.
#   conf=0.25   -> drop boxes the model is less than 25% sure about
#   imgsz=640   -> resize the image to 640 before looking (1280 finds smaller / far people, but slower)
#   verbose=False -> do not print extra text
# It returns a LIST with one "Results" object per image. We passed one image, so take item [0].
r = best.predict(IMAGE, conf=0.25, imgsz=640, verbose=False)[0]

# r.names is a dict that turns a class NUMBER into a NAME
print(r.names)       # {0: 'Helmet', 1: 'No-Helmet', 2: 'No-Vest', 3: 'Vest'}

# r.boxes.cls  = class number of every box     (a tensor, one value per box)
# r.boxes.xyxy = box corners in PIXELS         (one row per box: x1, y1, x2, y2)
# r.boxes.conf = confidence of every box       (0 to 1)
# zip() walks the three together; int(c) turns a tensor value into a number so we can look up the
# name; b.tolist() turns a tensor row into a normal Python list; float(s) makes the confidence a number.
items = [(r.names[int(c)], b.tolist(), float(s))
         for c, b, s in zip(r.boxes.cls, r.boxes.xyxy, r.boxes.conf)]

# print every detection: name, the box rounded to whole pixels, and the confidence
for name, box, score in items:
    print(name, [round(v) for v in box], round(score, 2))
# TRY: change conf to 0.1 and then 0.6 and see which boxes appear or disappear.


## 10. The full pipeline with the repo's own logic
*Why:* this connects the notebook to the repo. We clone the repo and import `verdict.py` (the Safe / Not-safe rule),
then run both models on the image and print one verdict per person. This is what `app.py` does, without the screen.

In [ ]:
# download the whole repo into Colab, then move into its folder (%cd changes the working folder)
!git clone https://github.com/suhaniranka006/helmet-vest-detection.git
%cd /content/helmet-vest-detection

import sys
# WHY: tell Python to also look in this folder when we write "import verdict"
sys.path.insert(0, ".")
from PIL import Image                                  # PIL opens and draws on images
# these three functions are the "thinking" part of the project (see verdict.py in the repo)
from verdict import assign_ppe, decide, overall

# model 1: MY fine-tuned model (Helmet / No-Helmet / Vest / No-Vest), the weights from the repo
ppe_model = YOLO("models/best_n.pt")
# model 2: the plain pretrained COCO model. We only use its class 0 = "person".
person_model = YOLO("yolov8n.pt")

# .convert("RGB") makes sure the image has 3 colour channels (PNGs can have transparency or be gray)
img = Image.open(IMAGE).convert("RGB")

# run both models on the same image
ppe = ppe_model.predict(img, conf=0.25, imgsz=640, verbose=False)[0]
# classes=[0] = keep only "person"; conf=0.40 = a person box must be at least 40% sure
per = person_model.predict(img, conf=0.40, imgsz=640, classes=[0], verbose=False)[0]

# turn model output into plain Python lists:
#   ppe_items = [("Helmet", [x1,y1,x2,y2]), ("Vest", [...]), ...]
#   persons   = [[x1,y1,x2,y2], [x1,y1,x2,y2], ...]
ppe_items = [(ppe_model.names[int(c)], b.tolist()) for c, b in zip(ppe.boxes.cls, ppe.boxes.xyxy)]
persons = [b.tolist() for b in per.boxes.xyxy]

# give each helmet/vest box to the person whose box contains the BOX CENTRE.
# Returns: one set of labels per person, and how many PPE boxes belonged to nobody (ignored).
labels_per_person, ignored = assign_ppe(ppe_items, persons)

verdicts = []
# enumerate(..., start=1) numbers the people 1, 2, 3...
for i, labels in enumerate(labels_per_person, start=1):
    # decide() applies the rule: any No-* -> NOT SAFE; Helmet+Vest -> SAFE; else UNCERTAIN
    verdict, why = decide(labels)
    verdicts.append(verdict)
    print(f"Person {i}: {verdict}  ({why})  labels={sorted(labels)}")

# the verdict for the whole photo: any NOT SAFE -> NOT SAFE; all SAFE -> SAFE; else UNCERTAIN
# (no person found -> "NO PERSON")
print("Whole photo:", overall(verdicts), "| ignored PPE boxes:", ignored)
# TRY: raise person conf to 0.9 and see what happens to the number of people.

## 11. Ground-truth evaluation (39 hand-labelled photos)
*Why:* mAP measures boxes, but users see verdicts. This runs the pipeline on the labelled photos and prints precision, recall
and the false-SAFE rate at two settings. Needs the repo clone from section 10.

In [ ]:
# eval/run_eval.py runs both models on the 39 labelled photos (eval/images) at two settings
# (640 px / person conf 0.40, and 1280 px / person conf 0.20), compares the verdicts with
# eval/ground_truth.csv, and prints the metrics (people recall, unsafe-photo precision and recall,
# false SAFE rate, ...). The ground truth is a COUNT per image, so the numbers are approximate.
!python eval/run_eval.py

## Where to go next
- `README.md` has the results, the ground-truth evaluation and the limitations.
- `verdict.py` is the Safe / Not-safe logic, `tests/` has its unit tests, `eval/` has the ground truth and metrics.
- Exercise: change `conf` in section 9 to 0.1 and 0.6, and see which boxes appear or disappear.